In [89]:
import os
import sys
import glob
import numpy as np
from tqdm import trange
from astropy.io import fits
from astropy.table import Table, vstack
from astropy.convolution import convolve, Gaussian1DKernel
import astropy.units as u
import astropy.coordinates as coord
import matplotlib
import matplotlib.pyplot as plt
from astropy.table import Column
from tqdm import trange
import pandas as pd
import fitsio
from astropy.table import Table, vstack
from astropy import units as u
from astropy.coordinates import SkyCoord
from easyquery import Query, QueryMaker
from scipy.stats import binomtest
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.colors import LogNorm
from matplotlib.colors import ListedColormap, BoundaryNorm
import h5py
from astropy.cosmology import Planck18

mpl.rcParams['font.family'] = 'serif'
mpl.rcParams['axes.linewidth'] = 1.5
mpl.rcParams['axes.xmargin'] = 1
mpl.rcParams['xtick.labelsize'] = 'x-large'
mpl.rcParams['xtick.major.size'] = 5
mpl.rcParams['xtick.major.width'] = 1.5
mpl.rcParams['ytick.labelsize'] = 'x-large'
mpl.rcParams['ytick.major.size'] = 5
mpl.rcParams['ytick.major.width'] = 1.5
mpl.rcParams['legend.frameon'] = False

rootdir = '/global/u1/v/virajvm/'
sys.path.append(os.path.join(rootdir, 'DESI2_LOWZ/desi_dwarfs/code'))

from desi_lowz_funcs import make_subplots, match_c_to_catalog, print_radecs, get_stellar_mass_mia
from desi_lowz_funcs import calc_normalized_dist
from desi_lowz_funcs import find_objects_nearby
# from construct_dwarf_galaxy_catalogs import process_sga_matches
from catalog_paper_plots import make_bar_pie


%load_ext autoreload
%autoreload 2


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [90]:
# Path to the catalog
filename = "/pscratch/sd/v/virajvm/desi_dwarf_catalogs/dr1/v1.0/desi_dr1_dwarf_catalog.fits"

# Option 1: load the MAIN extension directly as an Astropy Table
tot_cat = Table.read(filename, hdu="MAIN")
## we will cross match this with massive cat to remove those!!

In [91]:
hdus = fits.open("/pscratch/sd/v/virajvm/catalog_dr1_dwarfs/iron_bgs_bright_filter_zsucc_zrr02_allfracflux_INT.fits")

In [92]:
cat_all = hdus[1].data

In [93]:
ras = cat_all["RA"]
decs = cat_all["DEC"]
z = cat_all["Z"]
rmag = cat_all["MAG_R"]
gmag = cat_all["MAG_G"]
gr_col =  gmag - rmag

In [94]:
clean_mask = (cat_all["FRACFLUX_G"] < 0.35) & (cat_all["FRACFLUX_R"] < 0.35) & (cat_all["FRACFLUX_Z"] < 0.35)
zmask = (cat_all["Z"] < 0.15)

In [95]:
tot_mask = clean_mask & zmask

In [96]:
## estimate the stellar mass and get the non-dwarfs
mstar_vals = get_stellar_mass_mia( gr_col[tot_mask], gmag[tot_mask], z[tot_mask] )

In [97]:
mstar_mask = (mstar_vals > 10)

In [98]:
massive_galaxy_tab = Table({ "RA": ras[tot_mask][mstar_mask] , "DEC": decs[tot_mask][mstar_mask], "Z": z[tot_mask][mstar_mask], "MSTAR": mstar_vals[mstar_mask]  })



In [105]:
len(massive_galaxy_tab["RA"])

804010

In [106]:
comov_dist = Planck18.comoving_distance(massive_galaxy_tab["Z"]).value

In [107]:
massive_galaxy_tab["DIST_MPC"] = comov_dist

In [108]:
massive_galaxy_tab["MSTAR"].min()

10.000000256297703

In [109]:
out = Table()
out["ra"] = np.asarray(massive_galaxy_tab["RA"], dtype=np.float32)
out["dec"] = np.asarray(massive_galaxy_tab["DEC"], dtype=np.float32)
out["comov_dist"] = np.asarray(massive_galaxy_tab["DIST_MPC"], dtype=np.float32)

# Compute total size in bytes
total_bytes = sum(out[col].nbytes for col in out.colnames)
total_mb = total_bytes / (1024**2)
print(f"Total catalog size: {total_mb:.2f} MB")

out.write("/pscratch/sd/v/virajvm/catalog_dr1_dwarfs/massive_galaxy_slice_cat.csv", format="csv", overwrite=True)

Total catalog size: 9.20 MB
